In [1]:
import os
import numpy as np
import networkx as nx
from rdkit import Chem
from gspan import GSpan

In [8]:
print('Loading NCI dataset')
DATASET_DIR = "datasets/NCI_full"  # change this
graphs = []
y = []

filepath = "datasets/NCI_full/1total-connect.sdf"

supplier = Chem.SDMolSupplier(filepath, sanitize=False, removeHs=True)
for mol in supplier:
    if mol is None:
        continue

    G = nx.Graph()

    # Add atoms as nodes
    for atom in mol.GetAtoms():
        G.add_node(
            atom.GetIdx(),
            feature=atom.GetSymbol()   # WL uses node labels
        )

    # Add bonds as edges
    for bond in mol.GetBonds():
        G.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx(),
            bond_type=str(bond.GetBondType()),
            bond_order=bond.GetBondTypeAsDouble(),
            aromatic=bond.GetIsAromatic(),
            in_ring=bond.IsInRing(),
            conjugated=bond.GetIsConjugated(),
            stereo=str(bond.GetStereo())
        )

    # Get graph label
    # In NCI, class label is stored as a molecule property
    label = int(float(mol.GetProp("value")))
    graphs.append(G)
    y.append(label)

print(f"Loaded {len(graphs)} graphs")

Loading NCI dataset


[14:20:31] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[14:20:34] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[14:20:39] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[14:20:42] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[14:20:47] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[14:20:47] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.


Loaded 37349 graphs


In [9]:
# ===================================================================
# 2. Inspect edge label distribution (do this first)
# ===================================================================
from collections import Counter

bond_type_counts = Counter()
for G in graphs:
    for u, v, data in G.edges(data=True):
        bond_type_counts[data['bond_type']] += 1

print(f"\nBond type distribution:")
for bt, count in bond_type_counts.most_common():
    print(f"  {bt}: {count}")
print(f"  Unique bond types: {len(bond_type_counts)}")


Bond type distribution:
  SINGLE: 766387
  DOUBLE: 291974
  TRIPLE: 3958
  Unique bond types: 3


In [ ]:
# ===================================================================
# 3. Run gSpan
# ===================================================================

# --- Option A: bond_type as edge label (recommended) ---
# This matches the NCI literature: |LE| = 3 (SINGLE, DOUBLE, AROMATIC)
# and is what Thoma et al. (CORK, SDM 2009) used.

miner = GSpan(
    min_support=200,       # ~5% of dataset; tune as needed
    max_num_vertices=7,    # limit pattern size for tractability
    verbose=True
)
miner.run(
    graphs,
    node_label_attr='feature',
    edge_label_attr='bond_type',  # single attribute -> clean label space
)


[gSpan] Database: 37349 graphs, 64 vertex labels, 3 edge labels
[gSpan] Frequent 1-edge subgraphs: 28
  [gSpan] Found 1000 patterns so far (current code length: 5)
  [gSpan] Found 2000 patterns so far (current code length: 6)
  [gSpan] Found 3000 patterns so far (current code length: 6)
  [gSpan] Found 4000 patterns so far (current code length: 5)
  [gSpan] Found 5000 patterns so far (current code length: 6)
  [gSpan] Found 6000 patterns so far (current code length: 6)
  [gSpan] Found 7000 patterns so far (current code length: 6)
  [gSpan] Found 8000 patterns so far (current code length: 6)
  [gSpan] Found 9000 patterns so far (current code length: 6)
  [gSpan] Found 10000 patterns so far (current code length: 6)
  [gSpan] Found 11000 patterns so far (current code length: 5)
  [gSpan] Found 12000 patterns so far (current code length: 6)
  [gSpan] Found 13000 patterns so far (current code length: 6)
  [gSpan] Found 14000 patterns so far (current code length: 5)
  [gSpan] Found 15000 pat

In [ ]:
results = miner.get_frequent_subgraphs_as_nx()

print(f"\nTotal frequent subgraphs found: {len(results)}")
print(f"  Vertex labels mapped: {miner.vlabel_map}")
print(f"  Edge labels mapped:   {miner.elabel_map}")

In [ ]:
size_dist = Counter(r['num_vertices'] for r in results)
print(f"\nPattern size distribution:")
for s in sorted(size_dist):
    print(f"  |V|={s}: {size_dist[s]} patterns")

In [ ]:
print("\n--- Sample Frequent Subgraphs ---")
for i, r in enumerate(results[:15]):
    nodes = {n: d['feature'] for n, d in r['graph'].nodes(data=True)}
    edges = {(u, v): d['label'] for u, v, d in r['graph'].edges(data=True)}
    print(f"  #{i+1}: |V|={r['num_vertices']}, |E|={r['num_edges']}, "
          f"support={r['support']}")
    print(f"         nodes={nodes}")
    print(f"         edges={edges}")
    print(f"         DFS code: {r['dfs_code']}")

In [ ]:
# ===================================================================
# 5. Build binary indicator matrix (Def 2.3, CORK paper)
# ===================================================================
n_graphs = len(graphs)
n_features = len(results)

X = np.zeros((n_graphs, n_features), dtype=np.int8)
for feat_idx, r in enumerate(results):
    for gid in r['graph_ids']:
        X[gid, feat_idx] = 1

y_arr = np.array(y)

print(f"\nBinary indicator matrix: {X.shape}")
print(f"  Sparsity: {1 - X.mean():.4f}")
print(f"  Avg features per graph: {X.sum(axis=1).mean():.1f}")
print(f"  Class distribution: {dict(zip(*np.unique(y_arr, return_counts=True)))}")